### Import packages

In [2]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory, SolverStatus, TerminationCondition

### Create model

In [4]:
# Define model
m = ConcreteModel()

# Define sets
m.ITEMS = Set(initialize=['A', 'B', 'C'])
m.WEEKS = Set(initialize=range(1,5))

# Define Decision Variable
m.MarkPct = Var(m.ITEMS, m.WEEKS, domain = NonNegativeReals, bounds = (0,1))

# Define Implicit Variables
m.MarkPrice = Var(m.ITEMS, m.WEEKS, domain = NonNegativeReals)
m.Demand = Var(m.ITEMS, m.WEEKS, domain = NonNegativeReals)
m.Revenue = Var(m.ITEMS, m.WEEKS, domain = NonNegativeReals)
m.FinalInv = Var(m.ITEMS, m.WEEKS, domain = NonNegativeReals)

# Define Params
m.basePrice = Param(m.ITEMS, initialize = {'A':21.5, 'B':29.5, 'C': 30.5})
m.initInv = Param(m.ITEMS, initialize = {'A':10, 'B':30, 'C':41})
m.maxPct = 0.80
m.maxPctW1 = 0.20
m.cap = 20
m.dispCost = 1.2

# Define equations for implicit variables

# Define Markdown Price
def Define_MarkPrice_Rule(m,i,w):
    return (1-m.MarkPct[i,w])*m.basePrice[i] == m.MarkPrice[i,w]
m.Define_MarkPrice =  Constraint(m.ITEMS, m.WEEKS, rule = Define_MarkPrice_Rule)

# Define Demand (markdown % applied to the inventory available at the start of the week)
def Define_Demand_Rule(m,i,w):
    if w==1:
        return m.MarkPct[i,w]*m.initInv[i] == m.Demand[i,w]
    else:
        return m.MarkPct[i,w]*m.FinalInv[i,w-1] == m.Demand[i,w]
m.Define_Demand =  Constraint(m.ITEMS, m.WEEKS, rule = Define_Demand_Rule)

# Define Revenue
def Define_Revenue_Rule(m,i,w):
    return m.MarkPrice[i,w]*m.Demand[i,w] == m.Revenue[i,w]
m.Define_Revenue =  Constraint(m.ITEMS, m.WEEKS, rule = Define_Revenue_Rule)

# Define Final Inv (carried over from the previous week)
def Define_FinalInv_Rule(m,i,w):
    if w==1:
        return m.FinalInv[i,w] == m.initInv[i] - m.Demand[i,w]
    else:
        return m.FinalInv[i,w] == m.FinalInv[i,w-1] - m.Demand[i,w]
m.Define_FinalInv =  Constraint(m.ITEMS, m.WEEKS, rule = Define_FinalInv_Rule)


# Define Constraints

# Define max discount
def Max_Discount_Rule(m,i,w):
    return m.MarkPct[i,w] <= m.maxPct
m.Max_Discount = Constraint(m.ITEMS, m.WEEKS, rule = Max_Discount_Rule)

# Limit how deep the markdown can be in week 1 (made up in class)
def Limit_Markdown_Week1_Rule(m,i):
    return m.MarkPct[i,1] <= m.maxPctW1
m.Limit_Markdown_Week1 =  Constraint(m.ITEMS, rule = Limit_Markdown_Week1_Rule)

# Sell such that leftover inventory at the end of week 4 is at most cap
def Leftover_Inv_Rule(m,i):
    return m.FinalInv[i,4] <= m.cap
m.Leftover_Inv = Constraint(m.ITEMS, rule = Leftover_Inv_Rule)


# Define Objective Function
def Max_Revenue_Rule(m):
    return sum(m.Revenue[i,w] for i in m.ITEMS for w in m.WEEKS) - sum(m.dispCost*m.FinalInv[i,4] for i in m.ITEMS)
m.Max_Revenue = Objective(rule = Max_Revenue_Rule, sense = maximize)

### Solve

In [5]:
solver = SolverFactory('ipopt')
solution = solver.solve(m)

Check the status of the solver

In [6]:
print(solution['Solver'])


- Status: ok
  Message: Ipopt 3.14.20\x3a Optimal Solution Found
  Termination condition: optimal
  Id: 0
  Error rc: 0
  Time: 0.5214290618896484



### Extract Optimum Values

Print optimum value of objective function

In [7]:
print(m.Max_Revenue.expr())

1268.845259068645


Print optimum value of decision and implicit variables - Option 1 (tailored syntax, need to substitute the right indexes and variable names for other use cases)

In [ ]:
data = {'ITEM': [], 'WEEK': [], 'Opt_Discount': [], 'Opt_Price': [], 'Opt_Demand': [], 'Opt_Revenue': [], 'Final_Inv': []}
for i in m.ITEMS:
    for w in m.WEEKS:
        data['ITEM'].append(i)
        data['WEEK'].append(w)
        data['Opt_Discount'].append(m.MarkPct[i,w].value)
        data['Opt_Price'].append(m.MarkPrice[i,w].value)
        data['Opt_Demand'].append(m.Demand[i,w].value)
        data['Opt_Revenue'].append(m.Revenue[i,w].value)
        data['Final_Inv'].append(m.FinalInv[i,w].value)
pd.DataFrame(data)

Print optimum value of decision and implicit variables - Option 2 (generic, same syntax will work for other use cases)

In [ ]:
data = {'Variable': [], 'Value': []}
for var in m.component_objects(Var):
    for index in var:
        data['Variable'].append(f'{var.name}[{index}]')
        data['Value'].append(var[index].value)
pd.DataFrame(data)

### Abstract model

Create the input data as DataFrames (in practice, these would typically be read from CSV files or a database)

In [ ]:
# Item-level data: one row per item
items_df = pd.DataFrame({'ITEM': ['A', 'B', 'C'],
                         'basePrice': [21.5, 29.5, 30.5],
                         'initInv': [10, 30, 41]})

# Planning horizon: one row per week
weeks_df = pd.DataFrame({'WEEK': [1, 2, 3, 4]})

# Scalar parameters: one row per parameter
scalars_df = pd.DataFrame({'Param': ['maxPct', 'maxPctW1', 'cap', 'dispCost'],
                           'Value': [0.80, 0.20, 20, 1.2]})

print(items_df, weeks_df, scalars_df, sep='\n\n')

Define the abstract model: same formulation as above, but sets and parameters are declared without data. Since the weeks now come from data, we use `first()`, `prev()` and `last()` instead of hard-coding weeks 1 and 4

In [ ]:
# Define model
m = AbstractModel()

# Define sets (no data yet)
m.ITEMS = Set()
m.WEEKS = Set(ordered=True)

# Define Params (no data yet)
m.basePrice = Param(m.ITEMS)
m.initInv = Param(m.ITEMS)
m.maxPct = Param()
m.maxPctW1 = Param()
m.cap = Param()
m.dispCost = Param()

# Define Decision Variable
m.MarkPct = Var(m.ITEMS, m.WEEKS, domain = NonNegativeReals, bounds = (0,1))

# Define Implicit Variables
m.MarkPrice = Var(m.ITEMS, m.WEEKS, domain = NonNegativeReals)
m.Demand = Var(m.ITEMS, m.WEEKS, domain = NonNegativeReals)
m.Revenue = Var(m.ITEMS, m.WEEKS, domain = NonNegativeReals)
m.FinalInv = Var(m.ITEMS, m.WEEKS, domain = NonNegativeReals)

# Define equations for implicit variables

# Define Markdown Price
def Define_MarkPrice_Rule(m,i,w):
    return (1-m.MarkPct[i,w])*m.basePrice[i] == m.MarkPrice[i,w]
m.Define_MarkPrice =  Constraint(m.ITEMS, m.WEEKS, rule = Define_MarkPrice_Rule)

# Define Demand (markdown % applied to the inventory available at the start of the week)
def Define_Demand_Rule(m,i,w):
    if w == m.WEEKS.first():
        return m.MarkPct[i,w]*m.initInv[i] == m.Demand[i,w]
    else:
        return m.MarkPct[i,w]*m.FinalInv[i,m.WEEKS.prev(w)] == m.Demand[i,w]
m.Define_Demand =  Constraint(m.ITEMS, m.WEEKS, rule = Define_Demand_Rule)

# Define Revenue
def Define_Revenue_Rule(m,i,w):
    return m.MarkPrice[i,w]*m.Demand[i,w] == m.Revenue[i,w]
m.Define_Revenue =  Constraint(m.ITEMS, m.WEEKS, rule = Define_Revenue_Rule)

# Define Final Inv (carried over from the previous week)
def Define_FinalInv_Rule(m,i,w):
    if w == m.WEEKS.first():
        return m.FinalInv[i,w] == m.initInv[i] - m.Demand[i,w]
    else:
        return m.FinalInv[i,w] == m.FinalInv[i,m.WEEKS.prev(w)] - m.Demand[i,w]
m.Define_FinalInv =  Constraint(m.ITEMS, m.WEEKS, rule = Define_FinalInv_Rule)


# Define Constraints

# Define max discount
def Max_Discount_Rule(m,i,w):
    return m.MarkPct[i,w] <= m.maxPct
m.Max_Discount = Constraint(m.ITEMS, m.WEEKS, rule = Max_Discount_Rule)

# Limit how deep the markdown can be in the first week
def Limit_Markdown_Week1_Rule(m,i):
    return m.MarkPct[i,m.WEEKS.first()] <= m.maxPctW1
m.Limit_Markdown_Week1 =  Constraint(m.ITEMS, rule = Limit_Markdown_Week1_Rule)

# Sell such that leftover inventory at the end of the last week is at most cap
def Leftover_Inv_Rule(m,i):
    return m.FinalInv[i,m.WEEKS.last()] <= m.cap
m.Leftover_Inv = Constraint(m.ITEMS, rule = Leftover_Inv_Rule)


# Define Objective Function
def Max_Revenue_Rule(m):
    return sum(m.Revenue[i,w] for i in m.ITEMS for w in m.WEEKS) - sum(m.dispCost*m.FinalInv[i,m.WEEKS.last()] for i in m.ITEMS)
m.Max_Revenue = Objective(rule = Max_Revenue_Rule, sense = maximize)

Build the instance data from the DataFrames and create the model instance

In [ ]:
# Convert scalar parameters to a dictionary {Param: Value}
scalars = scalars_df.set_index('Param')['Value'].to_dict()

instanceData = {None: {
    'ITEMS': {None: items_df['ITEM'].tolist()},
    'WEEKS': {None: weeks_df['WEEK'].tolist()},
    'basePrice': items_df.set_index('ITEM')['basePrice'].to_dict(),
    'initInv': items_df.set_index('ITEM')['initInv'].to_dict(),
    'maxPct': {None: scalars['maxPct']},
    'maxPctW1': {None: scalars['maxPctW1']},
    'cap': {None: scalars['cap']},
    'dispCost': {None: scalars['dispCost']}}}

# Instantiate model
instance = m.create_instance(instanceData)

Solve the instance and check the status of the solver

In [ ]:
solver = SolverFactory('ipopt')
solution = solver.solve(instance)
print(solution['Solver'])

Print optimum value of objective function and of decision and implicit variables

In [ ]:
print(instance.Max_Revenue.expr())

In [ ]:
data = {'ITEM': [], 'WEEK': [], 'Opt_Discount': [], 'Opt_Price': [], 'Opt_Demand': [], 'Opt_Revenue': [], 'Final_Inv': []}
for i in instance.ITEMS:
    for w in instance.WEEKS:
        data['ITEM'].append(i)
        data['WEEK'].append(w)
        data['Opt_Discount'].append(instance.MarkPct[i,w].value)
        data['Opt_Price'].append(instance.MarkPrice[i,w].value)
        data['Opt_Demand'].append(instance.Demand[i,w].value)
        data['Opt_Revenue'].append(instance.Revenue[i,w].value)
        data['Final_Inv'].append(instance.FinalInv[i,w].value)
pd.DataFrame(data)